# Decision Tree Weather Type Classification

This notebook builds one model only: a **Decision Tree classifier** for the Kaggle Weather Type Classification dataset.  
It follows the same workflow as the Random Forest version so the group project remains consistent across team members.

The notebook includes:
- dataset loading
- feature/target split
- preprocessing with a scikit-learn pipeline
- Decision Tree training
- model evaluation
- cross-validation
- feature importance analysis
- final testing with new weather conditions


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

## 2. Load Dataset

In [ ]:
DATA_PATH = "weather_classification_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
display(df.info())
df.describe(include="all")

## 3. Split Features and Target

In [ ]:
TARGET_COLUMN = "Weather Type"

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COLUMN}', but found: {list(df.columns)}")

X = df.drop(columns=[TARGET_COLUMN])

# Encode the target weather condition labels as numbers.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET_COLUMN])
target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))

print("Feature columns:", list(X.columns))
print("Target encoding:", target_mapping)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 4. Preprocessing and Decision Tree Pipeline

Numeric columns use **median imputation** for missing values.  
Categorical columns use **most-frequent imputation** and **one-hot encoding**.  

Decision Trees do **not require feature scaling**, because they split data based on threshold values rather than distance calculations.


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
condition_features = [col for col in ["Cloud Cover", "Season", "Location"] if col in X.columns]

print("Numeric features:", numeric_features)
print("Categorical features encoded with OneHotEncoder:", categorical_features)
print("Condition features:", condition_features)

The next cell shows what condition encoding looks like.  
The model still uses the full preprocessing pipeline below, so encoding is learned properly from the training data only.


In [ ]:
if condition_features:
    encoded_condition_preview = pd.get_dummies(
        X[condition_features],
        columns=condition_features,
        dtype=int
    )
    display(encoded_condition_preview.head())
else:
    print("No condition columns found for encoding.")

In [ ]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        class_weight="balanced"
    ))
])

dt_pipeline

## 5. Train Model

In [ ]:
dt_pipeline.fit(X_train, y_train)

## 6. Evaluate Model

In [ ]:
y_pred = dt_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=range(len(target_encoder.classes_)))
cm_df = pd.DataFrame(cm, index=target_encoder.classes_, columns=target_encoder.classes_)
cm_df

## 7. Cross-Validation

In [ ]:
cv_scores = cross_val_score(dt_pipeline, X, y, cv=5, scoring="accuracy", n_jobs=1)

print("CV scores:", cv_scores)
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")

## 8. Feature Importance

Decision Trees provide feature importance values that show which variables had the strongest effect on the final splits.
Higher importance means the feature contributed more to the model's decision-making process.


In [ ]:
feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = dt_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
importance_df.head(20).plot(
    kind="barh",
    x="feature",
    y="importance",
    figsize=(10, 7),
    legend=False,
    title="Top 20 Decision Tree Feature Importances"
)

## 9. Final Test With New Weather Conditions

This final test creates new input rows using the same feature columns as the original dataset.
The pipeline automatically preprocesses and encodes the condition columns before predicting the final weather type.


In [ ]:
final_test_data = pd.DataFrame([
    {
        "Temperature": 5.0,
        "Humidity": 85,
        "Wind Speed": 12.0,
        "Precipitation (%)": 90.0,
        "Cloud Cover": "overcast",
        "Atmospheric Pressure": 1008.0,
        "UV Index": 1,
        "Season": "Winter",
        "Visibility (km)": 2.0,
        "Location": "inland"
    },
    {
        "Temperature": 31.0,
        "Humidity": 45,
        "Wind Speed": 4.0,
        "Precipitation (%)": 5.0,
        "Cloud Cover": "clear",
        "Atmospheric Pressure": 1020.0,
        "UV Index": 8,
        "Season": "Summer",
        "Visibility (km)": 10.0,
        "Location": "coastal"
    },
    {
        "Temperature": -6.0,
        "Humidity": 92,
        "Wind Speed": 18.0,
        "Precipitation (%)": 95.0,
        "Cloud Cover": "overcast",
        "Atmospheric Pressure": 995.0,
        "UV Index": 0,
        "Season": "Winter",
        "Visibility (km)": 1.0,
        "Location": "mountain"
    }
])

final_test_data = final_test_data[X.columns]
final_encoded_predictions = dt_pipeline.predict(final_test_data)
final_label_predictions = target_encoder.inverse_transform(final_encoded_predictions)

final_results = final_test_data.copy()
final_results["Predicted Weather Type"] = final_label_predictions
final_results

In [ ]:
final_probabilities = dt_pipeline.predict_proba(final_test_data)
probability_df = pd.DataFrame(
    final_probabilities,
    columns=[f"Probability: {label}" for label in target_encoder.classes_]
)

pd.concat([final_results[["Predicted Weather Type"]], probability_df], axis=1)